# Solving a Quadratic Unconstrained Binary Optimization (QUBO) problem on OVH

This notebook illustrates how to use the QUBO Solver to solve a QUBO problem:
- on a QPU, using the OVHConnection
- on a remote emulator, using the OVHConnection

We begin by installing the version of `qubo-solver` this notebook was written for, as its API may change between releases:

In [ ]:
%pip install qubo-solver==0.10.0

Then we import the relevant modules:

In [ ]:
import qoolqit
import torch
from pasqal_cloud import OVHConnection
from qoolqit.execution import QPU
from qubosolver import (
    Instance,
    QuantumSolvingConfig,
    RemoteEmulator,
    Solver,
    SolverConfig,
    analysis,
)

## Define the QUBO

A QUBO problem is defined by a symmetric matrix of coefficients:

In [ ]:
Q = torch.tensor([[-0.2, 0.0, 1.0], [0.0, -1.0, 1.5], [1.0, 1.5, -0.1]])
instance = Instance(Q)

## Connect to the Pasqal cloud

The OVHConnection can be used to access the Pasqal cloud platform, and fetch the available devices, and their associated QPUs.

In [ ]:
connection = OVHConnection()
connection.fetch_available_devices()

## Solve on a QPU

***This section runs on a real-hardware QPU.***

The solver post-processes the samples (`postprocessing=True`, also the default): every sampled bitstring is improved with a bit-flip local search, which flips each bit in turn and keeps the flip if it lowers the QUBO cost. The resulting probabilities therefore differ from the ones shown on the Pasqal Cloud portal. Set `postprocessing=False` to get the same results as the portal.

Pick the device you want to address. For this tutorial, we will pick Fresnel:

In [ ]:
device = qoolqit.Device.from_connection(connection, "FRESNEL_CAN1")
config = SolverConfig(
    solving=QuantumSolvingConfig(
        device=device, backend=QPU(connection=connection, num_shots=100)
    ),
    postprocessing=True,
)

solver = Solver(instance, config)
solutions = solver.solve()
df = analysis.to_dataframe([solutions])
df

## Solve on a remote emulator

Emulators are also available remotely via `pasqal_cloud`:

In [ ]:
config = SolverConfig(
    solving=QuantumSolvingConfig(backend=RemoteEmulator(connection=connection)),
    postprocessing=True,
)

solver = Solver(instance, config)
solutions = solver.solve()
df = analysis.to_dataframe([solutions])
df

## Plot the bitstring probabilities

In [ ]:
ax = (
    df.set_index("bitstrings")["probs"]
    .mul(100)
    .sort_values(ascending=False)
    .plot.bar(color="#4fab88", rot=60, ylabel="Probability [%]", xlabel="")
)